# v2a-inspect Client Demo

This notebook shows the basic client flow against a running `v2a-inspect-server`. Server requests are guarded by `RUN_SERVER_REQUESTS` so the notebook can be opened and run locally without accidentally starting expensive inference.

In [ ]:
from pathlib import Path

import httpx

from v2a_inspect.client import SAM3Client, VideoClient
from v2a_inspect.media_utils import probe_prepared_video
from v2a_inspect.models import VideoAsset
from v2a_inspect.preprocessing.scenes import detect_initial_scenes

In [ ]:
SERVER_URL = "http://localhost:8080"
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "test.mp4").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
VIDEO_PATH = (PROJECT_ROOT / "test.mp4").resolve()
RUN_SERVER_REQUESTS = True

VIDEO_PATH

## Local Video Inspection

The demo video is already in the prepared working-video format, so we can probe it and run local scene detection without contacting the server.

In [ ]:
probe = probe_prepared_video(VIDEO_PATH)
video_asset = VideoAsset(source_path=probe.path, frame_count=probe.frame_count)
initial_scenes = detect_initial_scenes(video_asset)
video_asset = video_asset.model_copy(update={"initial_scenes": initial_scenes})

{
    "source_path": str(video_asset.source_path),
    "frame_count": video_asset.frame_count,
    "duration_sec": video_asset.duration_sec,
    "scene_count": len(video_asset.initial_scenes),
}

In [ ]:
initial_scenes

## Select The Scene

The tracking request below uses the second detected scene, `initial_scenes[1]`. The API frame range is global and half-open: `[start_frame_index, end_frame_index)`. Returned track points also use global frame indexes.

In [ ]:
SCENE_INDEX = 3
scene = video_asset.initial_scenes[SCENE_INDEX]
START_FRAME_INDEX = scene.start_frame_index
# END_FRAME_INDEX = scene.end_frame_index
END_FRAME_INDEX = 1500
# FRAME_INDEX = (START_FRAME_INDEX + END_FRAME_INDEX) // 2
FRAME_INDEX = START_FRAME_INDEX + 10
TEXT_PROMPT = "airplane"

{
    "scene_index": SCENE_INDEX,
    "start_frame_index": START_FRAME_INDEX,
    "end_frame_index": END_FRAME_INDEX,
    "frame_count": scene.frame_count,
    "seed_frame_index": FRAME_INDEX,
    "text_prompt": TEXT_PROMPT,
}

## Server Health Check

Set `RUN_SERVER_REQUESTS = True` in the config cell when the server is running and you want to exercise the HTTP clients.

In [ ]:
if RUN_SERVER_REQUESTS:
    async with httpx.AsyncClient(base_url=SERVER_URL, timeout=30.0) as http_client:
        health = await http_client.get("/healthz")
        health.raise_for_status()
        print(health.json())
else:
    print("Skipped. Set RUN_SERVER_REQUESTS = True to call the server.")

## Upload Video

Client instances must be used as async context managers. This uploads the full source once; the tracking API can still restrict SAM3 inference to a frame range.

In [ ]:
video_id = None

if RUN_SERVER_REQUESTS:
    async with VideoClient(SERVER_URL) as video_client:
        upload_response = await video_client.upload(str(VIDEO_PATH))
        video_id = upload_response.video_id

video_id

## SAM3 Image Segmentation

This requests masks from the selected scene midpoint. Adjust `TEXT_PROMPT` for the object you want to segment.

In [ ]:
if RUN_SERVER_REQUESTS and video_id is not None:
    seed = SAM3Client.seed_from_prompt(TEXT_PROMPT)
    async with SAM3Client(SERVER_URL, timeout=300.0) as sam_client:
        segment_response = await sam_client.segment_image(
            video_id=video_id,
            frame_index=FRAME_INDEX,
            seeds=[seed],
            max_masks=3,
        )
    print(segment_response.model_dump())
else:
    print("Skipped. Upload a video first with RUN_SERVER_REQUESTS = True.")

## SAM3 Video Tracking: Second Scene Only

This tracks only `[START_FRAME_INDEX, END_FRAME_INDEX)`. The server decodes that frame range into a temporary frame directory, runs SAM3 on the bounded range, and returns global frame indexes. No video transcode is performed.

In [ ]:
RUN_TRACKING = True

if RUN_SERVER_REQUESTS and RUN_TRACKING and video_id is not None:
    seed = SAM3Client.seed_from_prompt(TEXT_PROMPT, frame_index=FRAME_INDEX)
    async with SAM3Client(SERVER_URL, timeout=900.0) as sam_client:
        tracking_response = await sam_client.track_video(
            video_id,
            seeds=[seed],
            start_frame_index=START_FRAME_INDEX,
            end_frame_index=END_FRAME_INDEX,
        )
    print({
        "track_count": len(tracking_response.tracks),
        "points_per_track": [len(track.points) for track in tracking_response.tracks],
        "first_frame_indexes": [
            track.points[0].frame_index for track in tracking_response.tracks if track.points
        ],
        "last_frame_indexes": [
            track.points[-1].frame_index for track in tracking_response.tracks if track.points
        ],
    })
else:
    print("Skipped. Set RUN_SERVER_REQUESTS = True and RUN_TRACKING = True to track video.")

## Visualization

These cells use `v2a_inspect.visualization` helpers to render scene summaries, segmented images, and tracking overlays for notebook inspection.

In [ ]:
from v2a_inspect.media_utils import extract_frame
from v2a_inspect.visualization import (
    display_image,
    display_video,
    render_scene_timeline,
    render_segmented_image,
    render_tracking_video,
    summarize_scenes,
    summarize_tracks,
)

In [ ]:
scene_timeline_image = render_scene_timeline(video_asset)
display_image(scene_timeline_image)
summarize_scenes(video_asset)[:5]

In [ ]:
if "segment_response" in globals():
    segment_frame = extract_frame(VIDEO_PATH, FRAME_INDEX)
    segmented_image = render_segmented_image(segment_frame, segment_response)
    display_image(segmented_image)
else:
    print("No segment_response available. Run the SAM3 image segmentation cell first.")

In [ ]:
if "tracking_response" in globals():
    visualization_dir = PROJECT_ROOT / "demo" / "outputs"
    tracking_overlay_path = visualization_dir / f"scene_{SCENE_INDEX:03d}_tracking_overlay.mp4"
    render_tracking_video(
        VIDEO_PATH,
        tracking_response,
        tracking_overlay_path,
        start_frame_index=START_FRAME_INDEX,
        end_frame_index=END_FRAME_INDEX,
    )
    display_video(tracking_overlay_path)
    summarize_tracks(tracking_response)
else:
    print("No tracking_response available. Run the SAM3 video tracking cell first.")

In [ ]:
tracking_overlay_path